# Week 1 - BTS Data Cleaning (Yihong Yu)

Goal: merge and clean 12 monthly BTS parquet files into one standardized table.

## Inputs
- `data/raw/bts/flights_2024_01.parquet` ... `data/raw/bts/flights_2024_12.parquet`

## Outputs
- `data/processed/bts/flights_2024_clean.parquet`
- `data/reports/bts/flights_2024_cancelled.parquet`
- `data/reports/bts/flights_2024_extreme.parquet`
- `data/reports/bts/bts_cleaning_summary_2024.csv`

## Notes
- Leakage-prone columns are intentionally retained at this stage for EDA.
- A model-ready feature table should be created in a later notebook.

## Step 0: Ensure project data (local or Hugging Face)

If `data/raw`, `data/processed`, and `data/reports` are missing or empty, the next cell downloads the team snapshot from the dataset repo [ahiruuu/CIS-5450](https://huggingface.co/datasets/ahiruuu/CIS-5450).

Set `FLIGHT_SKIP_HF=1` to never download (requires data already present). Set `FLIGHT_FORCE_HF=1` to re-download.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
for _p in [_here] + list(_here.parents):
    if (_p / "notebooks" / "project_data.py").exists():
        sys.path.insert(0, str(_p / "notebooks"))
        break

from project_data import ensure_project_data

ensure_project_data()

## Step 1: Environment and path setup (local + Colab compatible)

In [1]:
import os
from pathlib import Path

from project_data import resolve_project_root

# Optional Colab support
IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB and os.getenv("USE_COLAB_DRIVE", "0") == "1":
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')

PROJECT_ROOT = resolve_project_root()
DATA_ROOT = Path(os.getenv("FLIGHT_DATA_DIR", PROJECT_ROOT / "data")).expanduser().resolve()
RAW_DIR = DATA_ROOT / "raw" / "bts"
PROCESSED_DIR = DATA_ROOT / "processed" / "bts"
REPORT_DIR = DATA_ROOT / "reports" / "bts"

for p in [RAW_DIR, PROCESSED_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f"Project root:  {PROJECT_ROOT}")
print(f"Data root:     {DATA_ROOT}")
print(f"Raw input dir: {RAW_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"Reports dir:   {REPORT_DIR}")
print("Example files:", sorted([x.name for x in RAW_DIR.glob("flights_2024_*.parquet")])[:3])

Data root:     /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450/data
Raw input dir: /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450/data/raw
Processed dir: /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450/data/processed
Reports dir:   /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450/data/reports
Example files: ['flights_2024_01.parquet', 'flights_2024_02.parquet', 'flights_2024_03.parquet']


## Step 2: Merge 12 months and keep required columns

In [2]:
import numpy as np
import pandas as pd

# Keep project-relevant columns; drop diversion detail columns (Div1-5 etc.).
KEEP_COLS = [
    'FlightDate', 'Reporting_Airline', 'Tail_Number',
    'Flight_Number_Reporting_Airline',
    'Origin', 'Dest',
    'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15',
    'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15',
    'Cancelled', 'CancellationCode', 'Diverted',
    'CRSElapsedTime', 'ActualElapsedTime', 'AirTime',
    'Distance', 'TaxiOut', 'TaxiIn',
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay'
]

missing_months = []
dfs = []
for month in range(1, 13):
    path = RAW_DIR / f'flights_2024_{month:02d}.parquet'
    if not path.exists():
        missing_months.append(month)
        print(f'[MISSING] {path.name}')
        continue

    df_m = pd.read_parquet(path, columns=KEEP_COLS)
    dfs.append(df_m)
    print(f'2024-{month:02d}: {len(df_m):,} rows')

if missing_months:
    raise FileNotFoundError(f'Missing monthly parquet files: {missing_months}')

df = pd.concat(dfs, ignore_index=True)
initial_rows = len(df)
print(f'\nMerged rows: {initial_rows:,}')
print(f'Columns kept: {len(df.columns)}')

2024-01: 547,271 rows
2024-02: 519,221 rows
2024-03: 591,767 rows
2024-04: 582,185 rows
2024-05: 609,743 rows
2024-06: 611,132 rows
2024-07: 634,613 rows
2024-08: 619,025 rows
2024-09: 582,622 rows
2024-10: 615,497 rows
2024-11: 575,404 rows
2024-12: 590,581 rows

Merged rows: 7,079,061
Columns kept: 30


## Step 3: Filter to top-50 airports

In [3]:
TOP_50_AIRPORTS = [
    'ATL','LAX','ORD','DFW','DEN','JFK','SFO','SEA','LAS','MCO',
    'EWR','CLT','PHX','IAH','MIA','BOS','MSP','FLL','DTW','PHL',
    'LGA','BWI','SLC','DCA','SAN','MDW','TPA','HNL','PDX','STL',
    'DAL','BNA','AUS','IAD','OAK','MSY','RDU','SMF','SJC','SNA',
    'MCI','SAT','RSW','CLE','PIT','IND','CMH','OGG','ABQ','BUR'
]

before = len(df)
# Keep flights where Origin or Dest is in the top-50 list (route coverage).
df = df[df['Origin'].isin(TOP_50_AIRPORTS) | df['Dest'].isin(TOP_50_AIRPORTS)].copy()
print(f'Before filter: {before:,} rows')
print(f'After filter:  {len(df):,} rows (dropped {before - len(df):,})')
print(f'Unique origin airports: {df["Origin"].nunique()}')
print(f'Unique dest airports:   {df["Dest"].nunique()}')

Before filter: 7,079,061 rows
After filter:  6,938,563 rows (dropped 140,498)
Unique origin airports: 325
Unique dest airports:   325


## Step 4: Data profile and missingness snapshot

In [4]:
print('=== dtypes ===')
print(df.dtypes)
print('\n=== missing values ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct}).query('missing_count > 0')

=== dtypes ===
FlightDate                             str
Reporting_Airline                      str
Tail_Number                            str
Flight_Number_Reporting_Airline    float64
Origin                                 str
Dest                                   str
CRSDepTime                           int64
DepTime                            float64
DepDelay                           float64
DepDelayMinutes                    float64
DepDel15                           float64
CRSArrTime                           int64
ArrTime                            float64
ArrDelay                           float64
ArrDelayMinutes                    float64
ArrDel15                           float64
Cancelled                          float64
CancellationCode                       str
Diverted                           float64
CRSElapsedTime                     float64
ActualElapsedTime                  float64
AirTime                            float64
Distance                           floa

,missing_count,missing_pct
Tail_Number,18783,0.27
Flight_Number_Reporting_Airline,1,0.00
DepTime,90255,1.30
DepDelay,90544,1.30
DepDelayMinutes,90544,1.30
DepDel15,90544,1.30
ArrTime,95238,1.37
ArrDelay,110874,1.60
ArrDelayMinutes,110874,1.60
ArrDel15,110874,1.60


In [5]:
# Cancelled vs diverted vs operating flights
print('Cancelled flights:', df['Cancelled'].sum(), f"({df['Cancelled'].mean()*100:.2f}%)")
print('Diverted flights: ', df['Diverted'].sum(), f"({df['Diverted'].mean()*100:.2f}%)")
print('Operating (not cancelled):', (df['Cancelled'] == 0).sum())

Cancelled flights: 93809.0 (1.35%)
Diverted flights:  17065.0 (0.25%)
Operating (not cancelled): 6844754


## Step 5: Handle cancelled and diverted flights

Cancelled/diverted flights usually do not have complete realized delay fields. Archive them for EDA, then remove from the main modeling table.

In [6]:
# Save cancelled flights for EDA/audit
cancelled_path = REPORT_DIR / 'flights_2024_cancelled.parquet'
df_cancelled = df[df['Cancelled'] == 1].copy()
df_cancelled.to_parquet(cancelled_path, index=False)
print(f'Cancelled flights saved: {len(df_cancelled):,} rows -> {cancelled_path.name}')

# Main table keeps only non-cancelled and non-diverted flights
df = df[(df['Cancelled'] == 0) & (df['Diverted'] == 0)].copy()
print(f'Main table after filtering: {len(df):,} rows')

Cancelled flights saved: 93,809 rows -> flights_2024_cancelled.parquet
Main table after filtering: 6,827,689 rows


## Step 6: Handle critical missing values

In [7]:
before = len(df)

# Required for departure-delay task
required_cols = ['DepDelay', 'DepDel15']
df = df.dropna(subset=required_cols)
print(f"Dropped rows missing {required_cols}: {before - len(df):,}; remaining: {len(df):,}")

# Delay-cause fields are naturally null for many on-time flights; fill with 0 for consistency.
delay_cause_cols = ['CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']
df[delay_cause_cols] = df[delay_cause_cols].fillna(0)

# Tail_Number missing values are retained with sentinel value.
df['Tail_Number'] = df['Tail_Number'].fillna('UNKNOWN')

print('Delay-cause fields and Tail_Number handling completed.')

Dropped rows missing ['DepDelay', 'DepDel15']: 0; remaining: 6,827,689
Delay-cause fields and Tail_Number handling completed.


## Step 7: Outlier checks

In [8]:
print('DepDelay distribution:')
print(df['DepDelay'].describe(percentiles=[0.90, 0.95, 0.99, 0.999]))

# Archive and remove extreme positive departure delays (> 10 hours).
extreme_path = REPORT_DIR / 'flights_2024_extreme.parquet'
extreme_delay = df[df['DepDelay'] > 600].copy()
print(f'\nFlights with DepDelay > 600 minutes: {len(extreme_delay):,}')
extreme_delay.to_parquet(extreme_path, index=False)

before = len(df)
df = df[df['DepDelay'] <= 600].copy()
print(f'Remaining after outlier filter: {len(df):,} (removed {before - len(df):,})')

DepDelay distribution:
count    6.827689e+06
mean     1.261739e+01
std      5.566778e+01
min     -9.600000e+01
90%      4.300000e+01
95%      8.300000e+01
99%      2.150000e+02
99.9%    7.490000e+02
max      3.777000e+03
Name: DepDelay, dtype: float64

Flights with DepDelay > 600 minutes: 10,091
Remaining after outlier filter: 6,817,598 (removed 10,091)


In [9]:
# Validate CRSDepTime format as HHMM (allow 2400 as midnight edge case).
crs_dep = pd.to_numeric(df['CRSDepTime'], errors='coerce')
hh = (crs_dep // 100).astype('Int64')
mm = (crs_dep % 100).astype('Int64')

valid_time = (
    crs_dep.notna() &
    (
        ((hh >= 0) & (hh <= 23) & (mm >= 0) & (mm <= 59)) |
        (crs_dep == 2400)
    )
)

invalid_count = int((~valid_time).sum())
print(f'Invalid CRSDepTime rows: {invalid_count:,}')
if invalid_count > 0:
    df = df[valid_time].copy()
    print(f'Remaining after CRSDepTime filter: {len(df):,}')

Invalid CRSDepTime rows: 0


## Step 8: Type conversion and normalization

In [10]:
# Date field -> datetime
df['FlightDate'] = pd.to_datetime(df['FlightDate'], errors='coerce')

# Numeric fields -> nullable integer for safer conversion
int_cols = ['CRSDepTime', 'CRSArrTime', 'Distance']
for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

# Categorical fields
cat_cols = ['Reporting_Airline', 'Origin', 'Dest', 'CancellationCode']
for col in cat_cols:
    df[col] = df[col].astype('category')

# Target column as integer flag
df['DepDel15'] = pd.to_numeric(df['DepDel15'], errors='coerce').astype('Int64')

# Optional deduplication by key operational fields
before = len(df)
key_cols = ['FlightDate', 'Reporting_Airline', 'Flight_Number_Reporting_Airline', 'Origin', 'Dest', 'CRSDepTime']
df = df.drop_duplicates(subset=key_cols).copy()
print(f'Deduplicated rows: {before - len(df):,}')

print('Type conversion completed.')
print(df.dtypes)

Deduplicated rows: 0
Type conversion completed.
FlightDate                         datetime64[us]
Reporting_Airline                        category
Tail_Number                                   str
Flight_Number_Reporting_Airline           float64
Origin                                   category
Dest                                     category
CRSDepTime                                  Int64
DepTime                                   float64
DepDelay                                  float64
DepDelayMinutes                           float64
DepDel15                                    Int64
CRSArrTime                                  Int64
ArrTime                                   float64
ArrDelay                                  float64
ArrDelayMinutes                           float64
ArrDel15                                  float64
Cancelled                                 float64
CancellationCode                         category
Diverted                                  float64
CR

## Step 9: Cleaning quality report

In [11]:
print('=' * 60)
print('Cleaning quality report')
print('=' * 60)
print(f'Input rows:          {initial_rows:,}')
print(f'Rows after cleaning: {len(df):,}')
print(f'Retention:           {len(df) / initial_rows * 100:.1f}%')
print(f'Columns:             {len(df.columns)}')
print(f'Delay rate (>15m):   {df["DepDel15"].mean() * 100:.2f}%')
print(f'Mean DepDelay:       {df["DepDelay"].mean():.2f} min')
print(f'Date range:          {df["FlightDate"].min()} ~ {df["FlightDate"].max()}')
print('Remaining nulls (>0 only):')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('=' * 60)

summary_row = pd.DataFrame([
    {
        'year': 2024,
        'input_rows': int(initial_rows),
        'clean_rows': int(len(df)),
        'retention_pct': float(round(len(df) / initial_rows * 100, 3)),
        'n_columns': int(len(df.columns)),
        'delay_rate_pct': float(round(df['DepDel15'].mean() * 100, 4)),
        'mean_depdelay_min': float(round(df['DepDelay'].mean(), 4)),
        'date_min': str(df['FlightDate'].min()),
        'date_max': str(df['FlightDate'].max()),
    }
])
summary_path = REPORT_DIR / 'bts_cleaning_summary_2024.csv'
summary_row.to_csv(summary_path, index=False)
print(f'Saved summary csv: {summary_path}')

Cleaning quality report
Input rows:          7,079,061
Rows after cleaning: 6,817,598
Retention:           96.3%
Columns:             30
Delay rate (>15m):   20.49%
Mean DepDelay:       11.27 min
Date range:          2024-01-01 00:00:00 ~ 2024-12-31 00:00:00
Remaining nulls (>0 only):
Flight_Number_Reporting_Airline          1
CancellationCode                   6817598
dtype: int64


## Step 10: Save cleaned outputs

In [12]:
output_path = PROCESSED_DIR / 'flights_2024_clean.parquet'
df.to_parquet(output_path, index=False)

print(f'Saved cleaned table to: {output_path}')
print(f'File size: {output_path.stat().st_size / 1024 / 1024:.1f} MB')
print(f'Cancelled archive: {cancelled_path}')
print(f'Extreme-delay archive: {extreme_path}')

Saved cleaned table to: /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450/data/processed/flights_2024_clean.parquet
File size: 132.3 MB
Cancelled archive: /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450/data/reports/flights_2024_cancelled.parquet
Extreme-delay archive: /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450/data/reports/flights_2024_extreme.parquet
